# AI 面试高频手写代码速查（简洁实现 + 注释）

> 说明：以下实现以 **PyTorch + NumPy** 为主，突出“核心逻辑/公式”与“面试手写风格”。
> 代码为教学简化版，未做极致性能优化。

## 0. 公用工具

In [ ]:
import math
import torch
import torch.nn.functional as F


def stable_softmax(x, dim=-1):
    # 数值稳定 softmax: 先减去最大值，避免 exp 溢出
    x = x - x.max(dim=dim, keepdim=True).values
    return x.exp() / x.exp().sum(dim=dim, keepdim=True)

## 一、大模型注意力 / 编码模块

### 1.1 MHA / MQA / GQA（含复杂度提示）

In [ ]:
class SimpleMHA(torch.nn.Module):
    # 简化版 MHA
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.h = num_heads
        self.head_dim = dim // num_heads
        self.qkv = torch.nn.Linear(dim, dim * 3)
        self.out = torch.nn.Linear(dim, dim)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x)  # [B, T, 3C]
        q, k, v = qkv.chunk(3, dim=-1)
        # split heads
        q = q.view(B, T, self.h, self.head_dim).transpose(1, 2)  # [B, h, T, d]
        k = k.view(B, T, self.h, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.h, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, -1e9)
        attn = stable_softmax(attn, dim=-1)
        out = attn @ v  # [B, h, T, d]
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)


def mqa_attention(q, k, v, num_heads):
    # MQA：多头 Query，但共享 K/V（只 1 组）
    B, T, C = q.shape
    d = C // num_heads
    q = q.view(B, T, num_heads, d).transpose(1, 2)  # [B,h,T,d]
    # k,v 已是 [B, T, d]
    k = k.unsqueeze(1)  # [B,1,T,d]
    v = v.unsqueeze(1)
    attn = (q @ k.transpose(-2, -1)) / math.sqrt(d)
    attn = stable_softmax(attn, dim=-1)
    out = attn @ v  # [B,h,T,d]
    return out.transpose(1, 2).contiguous().view(B, T, C)


def gqa_attention(q, k, v, num_heads, num_kv_groups):
    # GQA：多个 Q 头共享较少的 K/V 组
    B, T, C = q.shape
    d = C // num_heads
    q = q.view(B, T, num_heads, d).transpose(1, 2)

    # k/v 按组展开，再复制到对应 Q 头
    k = k.view(B, T, num_kv_groups, d).transpose(1, 2)  # [B,g,T,d]
    v = v.view(B, T, num_kv_groups, d).transpose(1, 2)

    group_size = num_heads // num_kv_groups
    k = k.repeat_interleave(group_size, dim=1)  # [B,h,T,d]
    v = v.repeat_interleave(group_size, dim=1)

    attn = (q @ k.transpose(-2, -1)) / math.sqrt(d)
    attn = stable_softmax(attn, dim=-1)
    out = attn @ v
    return out.transpose(1, 2).contiguous().view(B, T, C)

### 1.2 FlashAttention（v1/v2 思想简化）

In [ ]:
# FlashAttention 核心思想：
# 分块计算 softmax，避免存储完整 [T,T] 注意力矩阵，实现“省显存”。
# 这里给出极简伪实现（非高性能），展示“分块 + 在线归一化”。


def flash_attention_blockwise(q, k, v, block_size=64):
    # q,k,v: [B, H, T, d]
    B, H, T, d = q.shape
    out = torch.zeros_like(q)

    for b in range(0, T, block_size):
        qb = q[:, :, b:b+block_size, :]  # [B,H,bs,d]
        # 维护每个 query 的最大值与分母（在线 softmax）
        m = torch.full((B, H, qb.shape[2], 1), -1e9, device=q.device)
        l = torch.zeros((B, H, qb.shape[2], 1), device=q.device)
        acc = torch.zeros_like(qb)

        for i in range(0, T, block_size):
            kb = k[:, :, i:i+block_size, :]
            vb = v[:, :, i:i+block_size, :]
            scores = (qb @ kb.transpose(-2, -1)) / math.sqrt(d)

            # 在线更新 m/l
            m_new = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)
            l = l * (m - m_new).exp() + (scores - m_new).exp().sum(dim=-1, keepdim=True)
            acc = acc * (m - m_new).exp() + (scores - m_new).exp() @ vb
            m = m_new

        out[:, :, b:b+block_size, :] = acc / l

    return out

### 1.3 位置编码：RoPE / ALiBi / Sinusoidal / T5 相对位置

In [ ]:
def sinusoidal_position_embedding(seq_len, dim):
    # 经典 Transformer 正弦位置编码
    pe = torch.zeros(seq_len, dim)
    pos = torch.arange(seq_len).unsqueeze(1)
    div = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe  # [T, C]


def rope_apply(q, k):
    # RoPE：旋转位置编码（简化版）
    B, H, T, d = q.shape
    assert d % 2 == 0
    freqs = torch.arange(0, d, 2, device=q.device) / d
    theta = 10000 ** (-freqs)
    positions = torch.arange(T, device=q.device)
    angles = positions[:, None] * theta[None, :]
    cos = torch.cos(angles)[None, None, :, :]
    sin = torch.sin(angles)[None, None, :, :]

    def rotate(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    return rotate(q), rotate(k)


def alibi_bias(num_heads, seq_len, device=None):
    # ALiBi：注意力线性偏置，无显式位置嵌入
    slopes = torch.tensor([2 ** (-8 * i / num_heads) for i in range(num_heads)], device=device)
    pos = torch.arange(seq_len, device=device)
    bias = pos[None, :] - pos[:, None]  # [T, T] 距离
    return slopes[:, None, None] * bias[None, :, :]  # [H, T, T]


def t5_relative_position_bias(num_heads, seq_len, num_buckets=32, max_distance=128, device=None):
    # T5 相对位置编码（bucket 化）
    pos = torch.arange(seq_len, device=device)
    rel = pos[None, :] - pos[:, None]  # [T,T]
    rel = rel.clamp(-max_distance, max_distance)
    buckets = ((rel + max_distance) * num_buckets / (2 * max_distance + 1)).long()
    bias_table = torch.zeros(num_heads, num_buckets, device=device)
    return bias_table[:, buckets]  # [H, T, T]

### 1.4 Masked Attention（padding mask / causal mask）

In [ ]:
def make_padding_mask(lengths, max_len):
    # lengths: [B]，返回 [B, 1, 1, T]
    B = lengths.size(0)
    idx = torch.arange(max_len, device=lengths.device)
    mask = idx[None, :] < lengths[:, None]
    return mask[:, None, None, :]


def make_causal_mask(seq_len, device=None):
    # 自回归 look-ahead mask
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask[None, None, :, :]  # [1,1,T,T]

### 1.5 Linear Attention（O(n)）

In [ ]:
def linear_attention(q, k, v, eps=1e-6):
    # 用可分解特征映射 phi(.) 代替 softmax
    phi = lambda x: F.elu(x) + 1
    q_ = phi(q)  # [B,H,T,d]
    k_ = phi(k)
    kv = k_.transpose(-2, -1) @ v  # [B,H,d,d]
    z = 1.0 / (q_ @ k_.sum(dim=-2, keepdim=True).transpose(-2, -1) + eps)
    out = (q_ @ kv) * z
    return out

### 1.6 Multi-Scale Attention（多窗口注意力思路）

In [ ]:
def multi_scale_attention(x, windows=(4, 8)):
    # 简化多尺度注意力：不同窗口大小局部 self-attention，最后融合。
    B, T, C = x.shape
    outs = []
    for w in windows:
        out = torch.zeros_like(x)
        for i in range(0, T, w):
            chunk = x[:, i:i+w, :]
            attn = stable_softmax(chunk @ chunk.transpose(-2, -1) / math.sqrt(C), dim=-1)
            out[:, i:i+w, :] = attn @ chunk
        outs.append(out)
    return torch.stack(outs, dim=0).mean(dim=0)

## 二、优化器 / 损失函数 / 归一化与嵌入

### 2.1 优化器：SGD/Momentum/Nesterov/AdamW/LAMB/LARS/RMSprop/Adagrad/Adadelta

In [ ]:
class SGD:
    def __init__(self, params, lr=1e-2):
        self.params = list(params)
        self.lr = lr

    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad


class SGDMomentum:
    def __init__(self, params, lr=1e-2, momentum=0.9, nesterov=False):
        self.params = list(params)
        self.lr = lr
        self.m = [torch.zeros_like(p) for p in self.params]
        self.mu = momentum
        self.nesterov = nesterov

    def step(self):
        for i, p in enumerate(self.params):
            self.m[i] = self.mu * self.m[i] + p.grad
            if self.nesterov:
                p.data -= self.lr * (self.mu * self.m[i] + p.grad)
            else:
                p.data -= self.lr * self.m[i]


class AdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        self.params = list(params)
        self.lr = lr
        self.b1, self.b2 = betas
        self.eps = eps
        self.wd = weight_decay
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            g = p.grad
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * (g * g)
            m_hat = self.m[i] / (1 - self.b1 ** self.t)
            v_hat = self.v[i] / (1 - self.b2 ** self.t)
            p.data = p.data * (1 - self.lr * self.wd) - self.lr * m_hat / (v_hat.sqrt() + self.eps)


def lars_update(p, g, lr=1e-3, eta=0.001, eps=1e-9):
    # LARS: layer-wise 自适应学习率
    w_norm = p.data.norm()
    g_norm = g.norm()
    local_lr = eta * w_norm / (g_norm + eps)
    return p.data - lr * local_lr * g


def lamb_update(p, g, m, v, t, lr=1e-3, b1=0.9, b2=0.999, eps=1e-6, wd=0.01):
    # LAMB: Adam + layer-wise 比例 (trust ratio)
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * (g * g)
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    update = m_hat / (v_hat.sqrt() + eps) + wd * p.data
    r = p.data.norm() / (update.norm() + eps)
    p.data = p.data - lr * r * update
    return m, v

### 2.2 学习率调度器：Step / Plateau / Cosine / Warmup

In [ ]:
class StepLR:
    def __init__(self, base_lr, step_size, gamma=0.1):
        self.base_lr = base_lr
        self.step_size = step_size
        self.gamma = gamma

    def __call__(self, step):
        return self.base_lr * (self.gamma ** (step // self.step_size))


class WarmupLR:
    def __init__(self, base_lr, warmup_steps):
        self.base_lr = base_lr
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        if step < self.warmup_steps:
            return self.base_lr * (step + 1) / self.warmup_steps
        return self.base_lr


def cosine_annealing_lr(base_lr, step, total_steps):
    return base_lr * 0.5 * (1 + math.cos(math.pi * step / total_steps))

### 2.3 损失函数：Label Smoothing / Focal / MSE / SmoothL1 / Triplet / Dice / KL / BCEWithLogits

In [ ]:
def label_smoothing_ce(logits, targets, smoothing=0.1):
    # Label smoothing: 防止过度自信
    n_classes = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(smoothing / (n_classes - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1 - smoothing)
    return (-true_dist * log_probs).sum(dim=1).mean()


def focal_loss(logits, targets, gamma=2.0, alpha=0.25):
    # Focal Loss: 处理类别不平衡
    probs = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probs, 1 - probs)
    w = alpha * (1 - pt) ** gamma
    return -(w * (targets * torch.log(probs + 1e-9) + (1 - targets) * torch.log(1 - probs + 1e-9))).mean()


def smooth_l1_loss(pred, target, beta=1.0):
    diff = (pred - target).abs()
    return torch.where(diff < beta, 0.5 * diff ** 2 / beta, diff - 0.5 * beta).mean()


def triplet_loss(anchor, pos, neg, margin=1.0):
    pos_dist = F.pairwise_distance(anchor, pos)
    neg_dist = F.pairwise_distance(anchor, neg)
    return F.relu(pos_dist - neg_dist + margin).mean()


def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    return 1 - (2 * intersection + eps) / (union + eps)


def kl_divergence(p_logits, q_logits):
    # KL(p||q)，常用于对齐 / 蒸馏
    p = F.log_softmax(p_logits, dim=-1)
    q = F.softmax(q_logits, dim=-1)
    return F.kl_div(p, q, reduction="batchmean")


def bce_with_logits(logits, targets):
    # 内置 sigmoid，数值更稳定
    return F.binary_cross_entropy_with_logits(logits, targets)

### 2.4 归一化 / 嵌入：LN / BN / WeightNorm / SpectralNorm / Token/Position/Patch

In [ ]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.gamma = torch.nn.Parameter(torch.ones(dim))
        self.beta = torch.nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def __call__(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.gamma = torch.nn.Parameter(torch.ones(dim))
        self.beta = torch.nn.Parameter(torch.zeros(dim))
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)
        self.eps = eps
        self.momentum = momentum

    def __call__(self, x, train=True):
        if train:
            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean = self.running_mean
            var = self.running_var
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta


def weight_norm(w, g, v):
    # Weight Norm: w = g * v / ||v||
    return g * v / (v.norm() + 1e-9)


def spectral_norm(w):
    # Spectral Norm: 使用最大奇异值约束
    u, s, v = torch.svd(w)
    return w / (s.max() + 1e-9)


def token_position_embedding(token_ids, vocab_size, dim):
    token_embed = torch.nn.Embedding(vocab_size, dim)
    pos_embed = sinusoidal_position_embedding(token_ids.size(1), dim).to(token_ids.device)
    return token_embed(token_ids) + pos_embed[None, :, :]


def patch_embedding(images, patch_size=16, dim=768):
    # ViT patch embedding: [B,C,H,W] -> [B, N, dim]
    B, C, H, W = images.shape
    assert H % patch_size == 0 and W % patch_size == 0
    patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    patches = patches.contiguous().view(B, C, -1, patch_size, patch_size)
    patches = patches.flatten(3)  # [B,C,N,patch_size*patch_size]
    patches = patches.transpose(1, 2).flatten(2)  # [B,N,C*patch^2]
    proj = torch.nn.Linear(C * patch_size * patch_size, dim)
    return proj(patches)

## 三、计算机视觉核心

### 3.1 IoU 系列（IoU/GIoU/DIoU/CIoU）

In [ ]:
def bbox_iou(box1, box2, eps=1e-9):
    # box: [x1,y1,x2,y2]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / (union + eps)


def giou(box1, box2):
    iou = bbox_iou(box1, box2)
    cx1 = min(box1[0], box2[0])
    cy1 = min(box1[1], box2[1])
    cx2 = max(box1[2], box2[2])
    cy2 = max(box1[3], box2[3])
    c_area = (cx2 - cx1) * (cy2 - cy1)
    inter_area = (min(box1[2], box2[2]) - max(box1[0], box2[0])) * (min(box1[3], box2[3]) - max(box1[1], box2[1]))
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter_area
    return iou - (c_area - union) / (c_area + 1e-9)


def diou(box1, box2):
    iou = bbox_iou(box1, box2)
    # center distance
    c1x = (box1[0] + box1[2]) / 2
    c1y = (box1[1] + box1[3]) / 2
    c2x = (box2[0] + box2[2]) / 2
    c2y = (box2[1] + box2[3]) / 2
    rho2 = (c1x - c2x) ** 2 + (c1y - c2y) ** 2
    # enclosing diagonal
    ex1 = min(box1[0], box2[0])
    ey1 = min(box1[1], box2[1])
    ex2 = max(box1[2], box2[2])
    ey2 = max(box1[3], box2[3])
    c2 = (ex2 - ex1) ** 2 + (ey2 - ey1) ** 2
    return iou - rho2 / (c2 + 1e-9)


def ciou(box1, box2):
    iou = bbox_iou(box1, box2)
    di = diou(box1, box2)
    # aspect ratio penalty
    w1, h1 = box1[2] - box1[0], box1[3] - box1[1]
    w2, h2 = box2[2] - box2[0], box2[3] - box2[1]
    v = (4 / (math.pi ** 2)) * (math.atan(w1 / (h1 + 1e-9)) - math.atan(w2 / (h2 + 1e-9))) ** 2
    alpha = v / (1 - iou + v + 1e-9)
    return di - alpha * v

### 3.2 NMS / Soft-NMS

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    # boxes: [N,4], scores: [N]
    idxs = scores.argsort(descending=True)
    keep = []
    while idxs.numel() > 0:
        i = idxs[0]
        keep.append(i.item())
        if idxs.numel() == 1:
            break
        ious = torch.tensor([bbox_iou(boxes[i].tolist(), boxes[j].tolist()) for j in idxs[1:]])
        idxs = idxs[1:][ious <= iou_threshold]
    return keep


def soft_nms(boxes, scores, sigma=0.5, iou_threshold=0.5, method="gaussian"):
    idxs = scores.argsort(descending=True)
    keep = []
    while idxs.numel() > 0:
        i = idxs[0]
        keep.append(i.item())
        if idxs.numel() == 1:
            break
        rest = idxs[1:]
        ious = torch.tensor([bbox_iou(boxes[i].tolist(), boxes[j].tolist()) for j in rest])
        if method == "linear":
            decay = torch.where(ious > iou_threshold, 1 - ious, torch.ones_like(ious))
        else:
            decay = torch.exp(-ious ** 2 / sigma)
        scores[rest] = scores[rest] * decay
        idxs = scores[rest].argsort(descending=True)
        idxs = rest[idxs]
    return keep

### 3.3 FPN / ROI Align / ResNet Block / Bottleneck

In [ ]:
class SimpleFPN(torch.nn.Module):
    # FPN: 上采样 + 侧边连接
    def __init__(self, c3, c4, c5, out=256):
        super().__init__()
        self.lat3 = torch.nn.Conv2d(c3, out, 1)
        self.lat4 = torch.nn.Conv2d(c4, out, 1)
        self.lat5 = torch.nn.Conv2d(c5, out, 1)

    def forward(self, C3, C4, C5):
        P5 = self.lat5(C5)
        P4 = self.lat4(C4) + F.interpolate(P5, scale_factor=2, mode="nearest")
        P3 = self.lat3(C3) + F.interpolate(P4, scale_factor=2, mode="nearest")
        return P3, P4, P5


def roi_align(feature, rois, output_size=(7,7)):
    # 简化 ROI Align：双线性插值
    out = []
    for roi in rois:
        x1, y1, x2, y2 = roi
        crop = feature[..., int(y1):int(y2), int(x1):int(x2)]
        out.append(F.interpolate(crop, size=output_size, mode="bilinear", align_corners=False))
    return torch.stack(out)


class ResBlock(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(dim, dim, 3, padding=1)
        self.conv2 = torch.nn.Conv2d(dim, dim, 3, padding=1)

    def forward(self, x):
        return x + self.conv2(F.relu(self.conv1(x)))


class BottleneckBlock(torch.nn.Module):
    def __init__(self, dim, bottleneck=64):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(dim, bottleneck, 1)
        self.conv2 = torch.nn.Conv2d(bottleneck, bottleneck, 3, padding=1)
        self.conv3 = torch.nn.Conv2d(bottleneck, dim, 1)

    def forward(self, x):
        y = F.relu(self.conv1(x))
        y = F.relu(self.conv2(y))
        y = self.conv3(y)
        return x + y

## 四、强化学习（DQN / A2C / DDPG / PPO / DPO）

In [ ]:
# DQN 核心伪实现（核心逻辑）
class DQN:
    def __init__(self, q_net, target_net, gamma=0.99):
        self.q = q_net
        self.q_t = target_net
        self.gamma = gamma

    def compute_loss(self, batch):
        states, actions, rewards, next_states, dones = batch
        q_vals = self.q(states).gather(1, actions)
        with torch.no_grad():
            max_next = self.q_t(next_states).max(dim=1, keepdim=True).values
            target = rewards + self.gamma * (1 - dones) * max_next
        return F.mse_loss(q_vals, target)


# A2C：优势 Actor-Critic
class A2C:
    def __init__(self, actor, critic, gamma=0.99):
        self.actor = actor
        self.critic = critic
        self.gamma = gamma

    def compute_loss(self, states, actions, rewards, next_states, dones):
        values = self.critic(states)
        next_values = self.critic(next_states).detach()
        td_target = rewards + self.gamma * (1 - dones) * next_values
        adv = td_target - values
        logp = F.log_softmax(self.actor(states), dim=-1).gather(1, actions)
        actor_loss = -(logp * adv.detach()).mean()
        critic_loss = F.mse_loss(values, td_target)
        return actor_loss + critic_loss

## 五、Transformer 核心架构（Encoder-Decoder / FFN / Gated FFN）

In [ ]:
class FFN(torch.nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1 = torch.nn.Linear(dim, hidden)
        self.fc2 = torch.nn.Linear(hidden, dim)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class GatedFFN(torch.nn.Module):
    # Gated FFN: GEGLU/SiGLU 思想
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc = torch.nn.Linear(dim, hidden * 2)
        self.proj = torch.nn.Linear(hidden, dim)

    def forward(self, x):
        h, gate = self.fc(x).chunk(2, dim=-1)
        return self.proj(F.gelu(gate) * h)


def split_heads(x, num_heads):
    B, T, C = x.shape
    d = C // num_heads
    return x.view(B, T, num_heads, d).transpose(1, 2)


def merge_heads(x):
    B, H, T, d = x.shape
    return x.transpose(1, 2).contiguous().view(B, T, H * d)

## 六、长文本 / 高效训练 / 工程实现

In [ ]:
def sliding_window_attention(x, window=128):
    # 滑动窗口注意力（局部）
    B, T, C = x.shape
    out = torch.zeros_like(x)
    for i in range(T):
        left = max(0, i - window)
        right = min(T, i + window + 1)
        chunk = x[:, left:right, :]
        qi = x[:, i:i+1, :]
        attn = stable_softmax(qi @ chunk.transpose(-2, -1) / math.sqrt(C), dim=-1)
        out[:, i:i+1, :] = attn @ chunk
    return out


def greedy_search(logits):
    return torch.argmax(logits, dim=-1)


def top_k_sampling(logits, k=5):
    topk = torch.topk(logits, k)
    probs = F.softmax(topk.values, dim=-1)
    idx = torch.multinomial(probs, 1)
    return topk.indices.gather(-1, idx)


def top_p_sampling(logits, p=0.9):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cumsum = torch.cumsum(probs, dim=-1)
    mask = cumsum <= p
    filtered = sorted_logits.masked_fill(~mask, -1e9)
    probs = F.softmax(filtered, dim=-1)
    idx = torch.multinomial(probs, 1)
    return sorted_idx.gather(-1, idx)

### 3.4 Anchor / Anchor-Free / U-Net / Conv2d 手写 / ViT Encoder

In [ ]:
# Anchor 生成（简化：单尺度）

def generate_anchors(feature_h, feature_w, stride=16, scales=(32, 64), ratios=(0.5, 1.0, 2.0)):
    anchors = []
    for y in range(feature_h):
        for x in range(feature_w):
            cx, cy = (x + 0.5) * stride, (y + 0.5) * stride
            for s in scales:
                for r in ratios:
                    w = s * math.sqrt(r)
                    h = s / math.sqrt(r)
                    anchors.append([cx - w/2, cy - h/2, cx + w/2, cy + h/2])
    return torch.tensor(anchors)


# Anchor-Free（CenterNet 思想：预测中心点 heatmap）

def centernet_target(gt_boxes, output_h, output_w):
    heatmap = torch.zeros(output_h, output_w)
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        heatmap[cy, cx] = 1.0
    return heatmap


# U-Net 简化版
class UNetBlock(torch.nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, out_ch, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(out_ch, out_ch, 3, padding=1),
            torch.nn.ReLU(),
        )

    def forward(self, x):
        return self.conv(x)


class SimpleUNet(torch.nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.enc1 = UNetBlock(in_ch, base)
        self.enc2 = UNetBlock(base, base*2)
        self.dec1 = UNetBlock(base*2 + base, base)
        self.pool = torch.nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        up = F.interpolate(e2, scale_factor=2, mode="nearest")
        d1 = self.dec1(torch.cat([up, e1], dim=1))
        return d1


# 纯 Python 版 Conv2d（单通道）

def conv2d_naive(x, kernel, stride=1, padding=0):
    # x: [H,W], kernel: [kH,kW]
    H, W = x.shape
    kH, kW = kernel.shape
    out_h = (H + 2*padding - kH) // stride + 1
    out_w = (W + 2*padding - kW) // stride + 1
    out = torch.zeros(out_h, out_w)

    x_pad = F.pad(x[None, None], (padding, padding, padding, padding)).squeeze()
    for i in range(out_h):
        for j in range(out_w):
            patch = x_pad[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = (patch * kernel).sum()
    return out


# ViT Encoder 简化版
class SimpleViTEncoder(torch.nn.Module):
    def __init__(self, dim=768, depth=2, heads=8):
        super().__init__()
        self.layers = torch.nn.ModuleList([
            torch.nn.TransformerEncoderLayer(d_model=dim, nhead=heads)
            for _ in range(depth)
        ])

    def forward(self, x):
        # x: [T,B,C] for torch Transformer
        for layer in self.layers:
            x = layer(x)
        return x

### 3.5 Focal + Dice 组合（分割常用）

In [ ]:
def focal_dice_loss(logits, targets, gamma=2.0, alpha=0.25):
    return focal_loss(logits, targets, gamma, alpha) + dice_loss(logits, targets)

## 四、强化学习补充：DDPG / TD3 / SAC / PPO / DPO / SFT / RM

In [ ]:
# DDPG：策略梯度 + Q 网络（简化）
class DDPG:
    def __init__(self, actor, critic, gamma=0.99):
        self.actor = actor
        self.critic = critic
        self.gamma = gamma

    def policy(self, s):
        return self.actor(s)

    def critic_loss(self, s, a, r, s2, done):
        q = self.critic(s, a)
        with torch.no_grad():
            a2 = self.actor(s2)
            q2 = self.critic(s2, a2)
            target = r + self.gamma * (1 - done) * q2
        return F.mse_loss(q, target)


def ppo_clip_loss(logits, old_logits, advantages, clip=0.2):
    ratio = torch.exp(F.log_softmax(logits, dim=-1) - F.log_softmax(old_logits, dim=-1))
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip, 1 + clip) * advantages
    return -torch.min(surr1, surr2).mean()


def dpo_loss(logp_chosen, logp_rejected, beta=0.1):
    # DPO: 直接偏好优化
    return -F.logsigmoid(beta * (logp_chosen - logp_rejected)).mean()


def sft_loss(logits, labels):
    # 监督微调 (SFT)
    return F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))


def rm_pairwise_loss(score_chosen, score_rejected):
    # 奖励模型：成对排序
    return -F.logsigmoid(score_chosen - score_rejected).mean()

## 七、工程细节：梯度裁剪 / 保存加载 / 梯度消失爆炸缓解

In [ ]:
# 梯度裁剪

def clip_grad(params, max_norm=1.0):
    torch.nn.utils.clip_grad_norm_(params, max_norm)


# 保存 / 加载

def save_model(model, path):
    torch.save(model.state_dict(), path)


def load_model(model, path):
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state)
    return model

### 1.7 Longformer Attention（局部 + 全局）

In [ ]:
def longformer_attention(x, window=4, global_idx=None):
    # 局部窗口 + 全局 token
    B, T, C = x.shape
    out = torch.zeros_like(x)
    global_idx = set(global_idx or [])
    for i in range(T):
        if i in global_idx:
            attn = stable_softmax(x[:, i:i+1, :] @ x.transpose(-2, -1) / math.sqrt(C), dim=-1)
            out[:, i:i+1, :] = attn @ x
        else:
            left, right = max(0, i-window), min(T, i+window+1)
            chunk = x[:, left:right, :]
            qi = x[:, i:i+1, :]
            attn = stable_softmax(qi @ chunk.transpose(-2, -1) / math.sqrt(C), dim=-1)
            out[:, i:i+1, :] = attn @ chunk
    return out

### 1.8 RoPE 插值 / 动态 RoPE

In [ ]:
def rope_interpolate(q, k, scale=2.0):
    # 通过缩放位置索引扩展上下文
    B, H, T, d = q.shape
    freqs = torch.arange(0, d, 2, device=q.device) / d
    theta = 10000 ** (-freqs)
    positions = torch.arange(T, device=q.device) / scale
    angles = positions[:, None] * theta[None, :]
    cos = torch.cos(angles)[None, None, :, :]
    sin = torch.sin(angles)[None, None, :, :]

    def rotate(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    return rotate(q), rotate(k)

### 3.6 EIoU / SIoU（简化）

In [ ]:
def eiou(box1, box2):
    iou = bbox_iou(box1, box2)
    w1, h1 = box1[2]-box1[0], box1[3]-box1[1]
    w2, h2 = box2[2]-box2[0], box2[3]-box2[1]
    cx1, cy1 = (box1[0]+box1[2])/2, (box1[1]+box1[3])/2
    cx2, cy2 = (box2[0]+box2[2])/2, (box2[1]+box2[3])/2
    cw = max(box1[2], box2[2]) - min(box1[0], box2[0])
    ch = max(box1[3], box2[3]) - min(box1[1], box2[1])
    center = ((cx1-cx2)**2 + (cy1-cy2)**2) / (cw**2 + ch**2 + 1e-9)
    wh = ((w1-w2)**2) / (cw**2 + 1e-9) + ((h1-h2)**2) / (ch**2 + 1e-9)
    return iou - center - wh


def siou(box1, box2):
    # SIoU: 角度/距离/形状综合（极简近似）
    return eiou(box1, box2)  # 近似占位

### 5.1 Transformer Encoder-Decoder（简化）

In [ ]:
class SimpleEncoder(torch.nn.Module):
    def __init__(self, dim=256, heads=4, depth=2):
        super().__init__()
        self.layers = torch.nn.ModuleList([
            torch.nn.TransformerEncoderLayer(d_model=dim, nhead=heads)
            for _ in range(depth)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class SimpleDecoder(torch.nn.Module):
    def __init__(self, dim=256, heads=4, depth=2):
        super().__init__()
        self.layers = torch.nn.ModuleList([
            torch.nn.TransformerDecoderLayer(d_model=dim, nhead=heads)
            for _ in range(depth)
        ])

    def forward(self, x, memory, tgt_mask=None, memory_mask=None):
        for layer in self.layers:
            x = layer(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)
        return x


class SimpleTransformer(torch.nn.Module):
    def __init__(self, dim=256, heads=4, depth=2):
        super().__init__()
        self.encoder = SimpleEncoder(dim, heads, depth)
        self.decoder = SimpleDecoder(dim, heads, depth)

    def forward(self, src, tgt, tgt_mask=None):
        memory = self.encoder(src)
        return self.decoder(tgt, memory, tgt_mask=tgt_mask)

### 6.1 LoRA / QLoRA / Adapter / Prefix / Prompt Tuning

In [ ]:
class LoRALinear(torch.nn.Module):
    def __init__(self, in_f, out_f, r=4, alpha=16):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.randn(out_f, in_f))
        self.lora_a = torch.nn.Parameter(torch.randn(r, in_f))
        self.lora_b = torch.nn.Parameter(torch.randn(out_f, r))
        self.scale = alpha / r

    def forward(self, x):
        base = F.linear(x, self.weight)
        lora = F.linear(x, self.lora_b @ self.lora_a) * self.scale
        return base + lora


def quantize_4bit(w):
    # QLoRA 简化：量化到 int8 模拟
    scale = w.abs().max() / 127
    q = (w / scale).round().clamp(-128, 127).to(torch.int8)
    return q, scale


def dequantize_4bit(q, scale):
    return q.float() * scale


class Adapter(torch.nn.Module):
    def __init__(self, dim, bottleneck=64):
        super().__init__()
        self.down = torch.nn.Linear(dim, bottleneck)
        self.up = torch.nn.Linear(bottleneck, dim)

    def forward(self, x):
        return x + self.up(F.relu(self.down(x)))


def prefix_tuning(prefix, hidden_states):
    # prefix: [P,C], hidden_states: [T,C]
    return torch.cat([prefix, hidden_states], dim=0)


def prompt_tuning(prompt_embed, token_embed):
    # 直接拼接可学习 prompt
    return torch.cat([prompt_embed, token_embed], dim=1)

### 6.2 Beam Search（简化）

In [ ]:
def beam_search_step(logits, beams, beam_size=3):
    # logits: [beam, vocab]
    scores = F.log_softmax(logits, dim=-1)
    all_candidates = []
    for i, (seq, score) in enumerate(beams):
        for token in range(scores.size(-1)):
            all_candidates.append((seq + [token], score + scores[i, token].item()))
    all_candidates.sort(key=lambda x: x[1], reverse=True)
    return all_candidates[:beam_size]

### 2.5 其他优化器：RMSprop / Adagrad / Adadelta

In [ ]:
class RMSprop:
    def __init__(self, params, lr=1e-3, alpha=0.99, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.alpha = alpha
        self.eps = eps
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        for i, p in enumerate(self.params):
            g = p.grad
            self.v[i] = self.alpha * self.v[i] + (1 - self.alpha) * (g * g)
            p.data -= self.lr * g / (self.v[i].sqrt() + self.eps)


class Adagrad:
    def __init__(self, params, lr=1e-2, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.h = [torch.zeros_like(p) for p in self.params]

    def step(self):
        for i, p in enumerate(self.params):
            g = p.grad
            self.h[i] += g * g
            p.data -= self.lr * g / (self.h[i].sqrt() + self.eps)


class Adadelta:
    def __init__(self, params, rho=0.95, eps=1e-6):
        self.params = list(params)
        self.rho = rho
        self.eps = eps
        self.eg = [torch.zeros_like(p) for p in self.params]
        self.edx = [torch.zeros_like(p) for p in self.params]

    def step(self):
        for i, p in enumerate(self.params):
            g = p.grad
            self.eg[i] = self.rho * self.eg[i] + (1 - self.rho) * (g * g)
            dx = (self.edx[i] + self.eps).sqrt() / (self.eg[i] + self.eps).sqrt() * g
            p.data -= dx
            self.edx[i] = self.rho * self.edx[i] + (1 - self.rho) * (dx * dx)

### 2.6 ReduceLROnPlateau（简化）

In [ ]:
class ReduceLROnPlateau:
    def __init__(self, base_lr, factor=0.5, patience=2, min_lr=1e-6):
        self.lr = base_lr
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.best = None
        self.bad = 0

    def step(self, metric):
        if self.best is None or metric < self.best:
            self.best = metric
            self.bad = 0
        else:
            self.bad += 1
            if self.bad >= self.patience:
                self.lr = max(self.lr * self.factor, self.min_lr)
                self.bad = 0
        return self.lr

### 3.7 Batch NMS / Fast NMS 思路

In [ ]:
# Batch NMS: 对不同类别做偏移，复用同一次 NMS

def batch_nms(boxes, scores, labels, iou_threshold=0.5, offset=4096):
    offsets = labels.float() * offset
    boxes_offset = boxes + offsets[:, None]
    return nms(boxes_offset, scores, iou_threshold)


# Fast NMS (简化): 先排序，批量计算 IoU，再阈值过滤

def fast_nms(boxes, scores, iou_threshold=0.5, top_k=200):
    idxs = scores.argsort(descending=True)[:top_k]
    boxes = boxes[idxs]
    iou_mat = torch.zeros((boxes.size(0), boxes.size(0)))
    for i in range(boxes.size(0)):
        for j in range(i+1, boxes.size(0)):
            iou_mat[i, j] = bbox_iou(boxes[i].tolist(), boxes[j].tolist())
    keep = (iou_mat.max(dim=0).values <= iou_threshold)
    return idxs[keep].tolist()

### 4.2 TD3 / SAC（核心思想简化）

In [ ]:
# TD3: 双 Q 网络 + 延迟策略更新（伪实现片段）
class TD3:
    def __init__(self, actor, critic1, critic2, gamma=0.99):
        self.actor = actor
        self.critic1 = critic1
        self.critic2 = critic2
        self.gamma = gamma

    def critic_loss(self, s, a, r, s2, done, policy_noise=0.2):
        with torch.no_grad():
            a2 = self.actor(s2) + policy_noise * torch.randn_like(a)
            q1 = self.critic1(s2, a2)
            q2 = self.critic2(s2, a2)
            target = r + self.gamma * (1 - done) * torch.min(q1, q2)
        q1 = self.critic1(s, a)
        q2 = self.critic2(s, a)
        return F.mse_loss(q1, target) + F.mse_loss(q2, target)


# SAC: 最大熵 RL（简化）

def sac_actor_loss(actor, critic, states, alpha=0.2):
    actions, logp = actor(states)  # 假设 actor 返回动作与 logp
    q = critic(states, actions)
    return (alpha * logp - q).mean()

### 4.3 MDP / Policy Gradient（简化）

In [ ]:
def discounted_returns(rewards, gamma=0.99):
    R = 0
    returns = []
    for r in reversed(rewards):
        R = r + gamma * R
        returns.append(R)
    return list(reversed(returns))


def policy_gradient_loss(log_probs, returns):
    # REINFORCE
    returns = torch.tensor(returns, device=log_probs.device)
    return -(log_probs * returns).mean()

### 4.4 IPO / SDPO / RAFT（占位实现思路）

In [ ]:
# IPO/SDPO/RAFT 一般是 DPO 的变体：
# 下面用统一接口表示“加权偏好损失”占位，面试可讲公式。

def preference_loss(logp_chosen, logp_rejected, beta=0.1, weight=1.0):
    return -weight * F.logsigmoid(beta * (logp_chosen - logp_rejected)).mean()